# 01 — Confirmatory unit-fresh dyad holdout

**Scope.** Read-only walkthrough of the frozen Sprint-A holdout artifacts. CPU only; no model fitting, no ICEWS reads, no row-level predictions.

**Design.** One-shot 20% unordered-dyad holdout covering 2018-01 to 2022-11. The split is **unit-fresh** (held-out dyads), **not time-fresh**: the model is trained on earlier months, but the evaluation period was retrospectively burned by earlier exploratory runs. The result is retrospective confirmatory evidence, not a prospective forecast.

**Models.**

- **49-feature logistic + temperature (primary).** The pre-registered primary specification.
- **29-feature logistic + temperature (secondary).** Pre-specified but outside the primary BH family; any advantage is a **post-hoc candidate**, not a new confirmatory test.
- **Warm-history Markov comparator.** Uses dyad counts observed only through 2017-11. This is the fair strong comparator. The train-only per-dyad Markov baseline collapses to the global transition on unit-fresh dyads.

**Claim boundary.** Dyad-level event propensity and ranking only. No node-level prediction claim, no causal claim, no actor-lifecycle claim. The within-dyad timing result is exploratory.


In [ ]:
from pathlib import Path
import ast
import json

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "results" / "framework_pilot").is_dir():
            return cand
    return here


ROOT = find_repo_root()
HOLDOUT = ROOT / "results" / "framework_pilot" / "holdout"
TABLES = HOLDOUT / "tables"


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


def load_json(path: Path) -> dict:
    if not path.is_file():
        print(f"[missing] {rel(path)}")
        return {}
    with path.open(encoding="utf-8") as fh:
        return json.load(fh)


def load_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        print(f"[missing] {rel(path)}")
        return pd.DataFrame()
    return pd.read_csv(path)


print("repo root:", ROOT)
print("holdout artifacts:", "present" if HOLDOUT.is_dir() else "MISSING")


## Frozen holdout summary

These are the design facts stored in `results/framework_pilot/holdout/summary.json`. They are displayed, not recomputed.


In [ ]:
summary = load_json(HOLDOUT / "summary.json")
holdout = summary.get("holdout", {})
verdict = summary.get("verdict", {})

if summary:
    facts = pd.DataFrame(
        [
            ("status", summary.get("status")),
            ("period", holdout.get("period")),
            ("rows", holdout.get("rows")),
            ("positives", holdout.get("positives")),
            ("prevalence", holdout.get("prevalence")),
            ("directed dyads", holdout.get("dyads")),
            ("unit-fresh", holdout.get("unit_fresh")),
            ("time-fresh", holdout.get("time_fresh")),
            ("claim level", verdict.get("claim_level")),
        ],
        columns=["field", "value"],
    )
    display(facts)
else:
    print("No summary available; later cells will report missing tables.")


## Table 2 Panel A — accuracy and calibration

Reconstructed from `tables/holdout_metrics_main.csv`. The global-transition and per-dyad Markov rows are identical by construction on unit-fresh dyads. Within-dyad AUC is shown under three definitions; the next section shows why the definition matters.


In [ ]:
metrics = load_csv(TABLES / "holdout_metrics_main.csv")

order = [
    "base_global_transition",
    "base_per_dyad_markov",
    "base_warm_markov",
    "logistic29_S1_temperature",
    "logistic49_S1_temperature",
    "hgb49_none",
]
labels = {
    "base_global_transition": "Global transition",
    "base_per_dyad_markov": "Per-dyad Markov",
    "base_warm_markov": "Warm-history Markov",
    "logistic29_S1_temperature": "Logistic29 + temperature",
    "logistic49_S1_temperature": "Logistic49 + temperature (primary)",
    "hgb49_none": "HGB49 (capacity control)",
}
cols = [
    "model", "log_loss", "brier", "ece", "auc", "ap",
    "within_auc_p1", "within_auc_strict_mean",
    "within_auc_strict_pooled",
    "calibration_slope", "calibration_intercept",
]

if not metrics.empty:
    panel_a = metrics[metrics["model"].isin(order)].copy()
    panel_a["_order"] = panel_a["model"].map(
        {m: i for i, m in enumerate(order)}
    )
    panel_a = panel_a.sort_values("_order").drop(columns="_order")
    panel_a["model"] = panel_a["model"].map(labels)
    panel_a = panel_a[cols].rename(
        columns={
            "log_loss": "LL",
            "brier": "Brier",
            "ece": "ECE",
            "auc": "AUC",
            "ap": "AP",
            "within_auc_p1": "Within P1",
            "within_auc_strict_mean": "Within strict mean",
            "within_auc_strict_pooled": "Within strict PW",
            "calibration_slope": "Slope",
            "calibration_intercept": "Intercept",
        }
    )
    numeric = [
        "LL", "Brier", "ECE", "AUC", "AP",
        "Within P1", "Within strict mean", "Within strict PW",
    ]
    panel_a[numeric] = panel_a[numeric].round(6)
    panel_a[["Slope", "Intercept"]] = (
        panel_a[["Slope", "Intercept"]].round(3)
    )
    panel_a = panel_a.reset_index(drop=True)
    display(panel_a)
else:
    print("Panel A not available.")


## Table 2 Panel C — primary versus warm history

Reconstructed from `tables/primary_comparisons.csv` and `tables/auc_ap_deltas.csv`. The difference is **primary − warm-history**. Positive log-loss/ECE means the primary is worse; positive AUC/AP means the primary is better.


In [ ]:
comparisons = load_csv(TABLES / "primary_comparisons.csv")
auc_ap = load_csv(TABLES / "auc_ap_deltas.csv")


def ci_pair(text):
    if isinstance(text, (list, tuple)) and len(text) == 2:
        return float(text[0]), float(text[1])
    try:
        lo, hi = json.loads(text)
        return float(lo), float(hi)
    except Exception:
        return float("nan"), float("nan")


warm_key = "logistic49_temperature__vs__warm_markov"
warm = comparisons[comparisons["comparison"] == warm_key]
warm_auc = auc_ap[auc_ap["comparison"] == warm_key]

if metrics.empty or warm.empty or warm_auc.empty:
    print("Panel C not available (missing metrics or warm-history rows).")
else:
    m = metrics.set_index("model")
    needed = {"logistic49_S1_temperature", "base_warm_markov"}
    if not needed.issubset(set(m.index)):
        print("Panel C not available; missing metric rows.")
    else:
        primary = m.loc["logistic49_S1_temperature"]
        warmm = m.loc["base_warm_markov"]
        w = warm.iloc[0]
        a = warm_auc.iloc[0]

        ll_lo, ll_hi = ci_pair(w["multiway_ci_ll"])
        ece_lo, ece_hi = ci_pair(w["multiway_ci_ece"])
        auc_lo, auc_hi = ci_pair(a["d_auc_multiway_ci"])
        ap_lo, ap_hi = ci_pair(a["d_ap_multiway_ci"])

        panel_c = pd.DataFrame(
            [
                (
                    "Log-loss",
                    primary["log_loss"],
                    warmm["log_loss"],
                    -w["d_log_loss"],
                    (-ll_hi, -ll_lo),
                    "primary worse" if w["d_log_loss"] < 0 else "primary better",
                ),
                (
                    "ECE",
                    primary["ece"],
                    warmm["ece"],
                    -w["d_ece"],
                    (-ece_hi, -ece_lo),
                    "primary worse" if w["d_ece"] < 0 else "primary better",
                ),
                (
                    "AUC",
                    primary["auc"],
                    warmm["auc"],
                    a["d_auc"],
                    (auc_lo, auc_hi),
                    "primary better" if a["d_auc"] > 0 else "primary worse",
                ),
                (
                    "AP",
                    primary["ap"],
                    warmm["ap"],
                    a["d_ap"],
                    (ap_lo, ap_hi),
                    "primary better" if a["d_ap"] > 0 else "primary worse",
                ),
            ],
            columns=[
                "Metric", "Primary", "Warm-history",
                "Difference (primary - warm)", "95% CI", "Direction",
            ],
        )
        for col in ["Primary", "Warm-history", "Difference (primary - warm)"]:
            panel_c[col] = panel_c[col].astype(float).round(6)
        panel_c["95% CI"] = panel_c["95% CI"].apply(
            lambda t: f"[{t[0]:.6f}, {t[1]:.6f}]"
        )
        display(panel_c)


## Within-dyad definitions: strict mean versus pair-weighted

The three within-dyad AUCs answer different questions:

- **P1-compatible centred:** subtract each dyad's mean score, then pool.
- **Strict mean:** compute AUC inside each eligible dyad, then give every dyad equal weight.
- **Strict pair-weighted:** pool within-dyad pairs, so dyads with many pairs dominate.

Against per-dyad Markov, the strict mean is essentially unchanged (Δ = −0.0047; CI includes 0), while the pair-weighted version improves (+0.0240). The gain is therefore concentrated in dyads with many within-dyad pairs, not broad across the typical dyad. The P1-compatible centred contrast is also null (Δ = −0.0049; CI includes 0).


In [ ]:
within = load_csv(TABLES / "within_between_auc.csv")
deltas = load_csv(TABLES / "within_between_deltas.csv")

model_order = [
    "logistic49_S1_temperature",
    "logistic29_S1_temperature",
    "base_per_dyad_markov",
    "hgb49_none",
]
model_labels = {
    "logistic49_S1_temperature": "Logistic49 (primary)",
    "logistic29_S1_temperature": "Logistic29 (secondary)",
    "base_per_dyad_markov": "Per-dyad Markov",
    "hgb49_none": "HGB49",
}

if not within.empty:
    wtab = within[within["model"].isin(model_order)].copy()
    wtab["_order"] = wtab["model"].map(
        {m: i for i, m in enumerate(model_order)}
    )
    wtab = wtab.sort_values("_order").drop(columns="_order")
    wtab["model"] = wtab["model"].map(model_labels)
    wtab = wtab[
        [
            "model", "p1_within_auc", "mean", "mean_ci",
            "pooled", "pooled_ci", "n_dyads_eligible",
        ]
    ].rename(
        columns={
            "model": "Model",
            "p1_within_auc": "Within P1",
            "mean": "Strict mean",
            "mean_ci": "Strict mean 95% CI",
            "pooled": "Strict pair-weighted",
            "pooled_ci": "Pair-weighted 95% CI",
            "n_dyads_eligible": "Eligible dyads",
        }
    )
    for col in ["Within P1", "Strict mean", "Strict pair-weighted"]:
        wtab[col] = wtab[col].astype(float).round(6)
    wtab = wtab.reset_index(drop=True)
    display(wtab)
else:
    print("Within/between table not available.")

pair_key = "logistic49_S1_temperature__vs__base_per_dyad_markov"
if not deltas.empty:
    row = deltas[deltas["comparison"] == pair_key]
    if row.empty:
        print("Paired within-dyad contrast not available.")
    else:
        r = row.iloc[0]
        centered = r["centered"]
        if isinstance(centered, str):
            try:
                centered = ast.literal_eval(centered)
            except Exception:
                centered = {}
        contrast = pd.DataFrame(
            [
                (
                    "Strict mean (equal dyad weight)",
                    r["d_mean"], r["d_mean_ci"], r["p_mean"],
                ),
                (
                    "Strict pair-weighted",
                    r["d_pooled"], r["d_pooled_ci"], r["p_pooled"],
                ),
                (
                    "P1-compatible centred",
                    centered.get("d"), centered.get("ci"),
                    centered.get("p"),
                ),
            ],
            columns=["Definition", "Delta AUC (primary - Markov)", "95% CI", "p"],
        )
        contrast["Delta AUC (primary - Markov)"] = pd.to_numeric(
            contrast["Delta AUC (primary - Markov)"], errors="coerce"
        ).round(6)
        display(contrast)


## Graph and train-statistics leave-one-group-out

`delta_vs_full` is log-loss **without the group minus the full model**. Positive means the group carries held-out signal; negative means removing the group improves the reduced model. The graph block is negative (−0.0129), as is train statistics (−0.0240). The 20-feature graph block does not transfer to the holdout. This is a negative result about that pre-specified block, not graph features in general.


In [ ]:
loo = load_csv(TABLES / "feature_group_loo.csv")

if not loo.empty:
    ltab = loo[loo["group"].isin(["graph", "train_stats"])].copy()
    ltab["Features removed"] = 49 - ltab["n_features"]
    ltab = ltab[
        ["group", "n_features", "Features removed", "delta_vs_full", "ci", "p"]
    ]
    ltab = ltab.rename(
        columns={
            "group": "Feature group",
            "n_features": "Features remaining",
            "delta_vs_full": "Delta LL without group",
            "ci": "95% dyad bootstrap CI",
            "p": "p",
        }
    )
    ltab["Delta LL without group"] = (
        ltab["Delta LL without group"].astype(float).round(6)
    )
    ltab = ltab.reset_index(drop=True)
    display(ltab)
else:
    print("Feature-group LOO table not available.")


**Figure note.** The manuscript's Figure 3 (`paper/figures/fig3_holdout.png` in the manuscript source) is **not** redistributed with the public repository, because `paper/` is excluded. Panel (a) is reconstructed by the forest below from `tables/primary_comparisons.csv`; panel (b) by the LOO table above; panel (c) by the within-dyad table. No paper file is read.


In [ ]:
forest_keys = [
    "logistic49_temperature__vs__per_dyad_markov",
    "logistic49_temperature__vs__dyad_base_rate",
    "logistic49_temperature__vs__markov_recency",
    "logistic49_temperature__vs__warm_markov",
]
forest_labels = {
    "logistic49_temperature__vs__per_dyad_markov": "Per-dyad Markov",
    "logistic49_temperature__vs__dyad_base_rate": "Dyad base rate",
    "logistic49_temperature__vs__markov_recency": "Markov recency",
    "logistic49_temperature__vs__warm_markov": "Warm-history Markov",
}

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print(f"matplotlib unavailable: {exc}")

if plt is not None and not comparisons.empty:
    f = comparisons[comparisons["comparison"].isin(forest_keys)].copy()
    f["_order"] = f["comparison"].map(
        {k: i for i, k in enumerate(forest_keys)}
    )
    f = f.sort_values("_order").drop(columns="_order")
    missing = [k for k in forest_keys if k not in set(f["comparison"])]
    if missing:
        print("Missing forest rows:", missing)

    xs = f["d_log_loss"].astype(float).to_numpy()
    bounds = [ci_pair(v) for v in f["multiway_ci_ll"]]
    los = [b[0] for b in bounds]
    his = [b[1] for b in bounds]

    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    ypos = list(range(len(f)))
    ax.errorbar(
        xs, ypos,
        xerr=[
            [x - lo for x, lo in zip(xs, los)],
            [hi - x for x, hi in zip(xs, his)],
        ],
        fmt="o", color="#1f4e79", ecolor="#7f7f7f",
        capsize=3, markersize=5,
    )
    ax.axvline(0, color="#999999", linewidth=0.8)
    ax.set_yticks(ypos)
    ax.set_yticklabels(
        [forest_labels[k] for k in f["comparison"]]
    )
    ax.invert_yaxis()
    ax.set_xlabel(
        "Delta log-loss (comparator - primary; positive = primary better)"
    )
    ax.set_title("Holdout contrasts (frozen aggregates)")
    fig.tight_layout()
    plt.show()
elif plt is None:
    print("Forest skipped; install matplotlib to render it.")
else:
    print("Forest skipped; comparison table not available.")


## Honest reading

- **Against the degenerate per-dyad Markov baseline**, the primary model looks strong: ΔLL +0.0480 (+18.05%), ΔAUC +0.1779, ΔAP +0.3507. That comparison is necessary but not sufficient, because the baseline has no training rows for held-out dyads.
- **Against warm history**, the primary improves ranking (AUC +0.0160, AP +0.0350) but loses on log-loss (+0.0081) and ECE (+0.0319). The proper scores establish the substantive loss. ECE alone is not decisive because the warm baseline is low-sharpness, and the primary's calibration slope is closer to one (0.120 vs 0.071); the calibration evidence is mixed, not uniformly worse.
- **The 29-feature model** is a post-hoc candidate for dyad-level ranking with a leaner score: log-loss 0.2052, ECE 0.0275, AUC 0.9083, AP 0.6717, versus warm history 0.2097 / 0.0102 / 0.8942 / 0.6368. Its strict mean within-dyad AUC is 0.4716, so this LL gain must **not** be read as actor-level explanation or timing. It is secondary, not in a BH family, and not a new confirmatory test.
- **No new confirmatory test is run here.** The holdout labels were read once; this notebook re-displays frozen aggregates only. No node-level claim is made.
